[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fisica_Atomica_y_Molecular_(AMO)/AMO-11_Interaccion_Luz_Materia_Simulacion.ipynb)



# Simulación Computacional: Interaccion Luz Materia
**Área:** Fisica Atomica y Molecular

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

def optical_bloch(state, t, Omega, Delta, Gamma, gamma_d):
    """
    Ecuaciones de Bloch ópticas para la matriz densidad de 2 niveles.
    state = [rho_gg, rho_ee, Re(rho_ge), Im(rho_ge)]
    """
    r_gg, r_ee, r_ge_real, r_ge_imag = state
    r_ge = r_ge_real + 1j * r_ge_imag
    r_eg = np.conj(r_ge)
    
    # Dinámica de poblaciones
    dr_gg = Gamma * r_ee + 1j * (Omega/2.0) * (r_eg - r_ge)
    dr_ee = -Gamma * r_ee - 1j * (Omega/2.0) * (r_eg - r_ge)
    
    # Dinámica de coherencia (decaimiento total = Gamma/2 + gamma_d)
    dr_ge = -(Gamma/2.0 + gamma_d + 1j*Delta) * r_ge - 1j * (Omega/2.0) * (r_ee - r_gg)
    
    return [np.real(dr_gg), np.real(dr_ee), np.real(dr_ge), np.imag(dr_ge)]

# Parámetros experimentales
Omega = 10.0      # Frecuencia de Rabi (fuerza del láser)
Delta = 0.0       # Láser en estricta resonancia
Gamma = 1.0       # Tasa de decaimiento espontáneo poblacional
gamma_d = 0.5     # Tasa de decaimiento de desfasamiento extra (colisiones)

t_span = np.linspace(0, 5.0, 500)
# Inicialmente todo en el estado base |g>
initial_state = [1.0, 0.0, 0.0, 0.0]

# Resolución numérica
solution = odeint(optical_bloch, initial_state, t_span, args=(Omega, Delta, Gamma, gamma_d))

rho_gg = solution[:, 0]
rho_ee = solution[:, 1]
coherence_abs = np.sqrt(solution[:, 2]**2 + solution[:, 3]**2)

plt.figure(figsize=(10, 6))
plt.plot(t_span, rho_ee, 'r-', lw=2, label='Población Excitada $\\rho_{ee}$')
plt.plot(t_span, rho_gg, 'b--', lw=2, label='Población Base $\\rho_{gg}$')
plt.plot(t_span, coherence_abs, 'g-', lw=2, label='Magnitud de Coherencia $|\\rho_{ge}|$')

plt.title("Dinámica de Ecuaciones de Bloch Ópticas con Disipación")
plt.xlabel("Tiempo (en unidades de $1/\\Gamma$)")
plt.ylabel("Elementos de la Matriz Densidad")
plt.axhline(0.5, color='gray', linestyle=':', label='Límite de Saturación (0.5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
# plt.show()